# Install Dependencies

In [ ]:
!pip install pytesseract
!pip install pdfplumber
!pip install pdf2image
!pip install -q rake-nltk rouge-score
!pip install -q transformers torch scikit-learn
!pip install -q scispacy

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 22.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0, but you have numpy 1.

In [ ]:
# Install compatible scispaCy model (v0.5.4 works with spacy>=3.7)
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz || echo 'scispacy model install failed, will use keyword fallback'

In [ ]:
# Poppler (needed by pdf2image on Linux/Colab)
!apt-get install -q poppler-utils tesseract-ocr

# **Libraries**

In [ ]:
import os
import re
import string
import nltk
import torch
import numpy as np

from PIL import Image
import pytesseract
import pdfplumber
from pdf2image import convert_from_path

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

from rake_nltk import Rake
from rouge_score import rouge_scorer
from transformers import AutoTokenizer, AutoModel, pipeline
from sklearn.metrics.pairwise import cosine_similarity

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# Download NLTK resources silently
for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(resource, quiet=True)

print(' All imports successful')

# **File type**

In [ ]:
import os

def get_file_type(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext in ['.png', '.jpg', '.jpeg']:
        return "image"
    elif ext == '.pdf':
        return "pdf"
    else:
        return "unsupported"

# **Image → Text (OCR)**

In [ ]:
def extract_from_image(image_path):
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)
    return text

# **PDF → Text**

In [ ]:
def extract_text_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

In [ ]:
def extract_text_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

# **Scanned PDF → Text**

In [ ]:
def extract_scanned_pdf(pdf_path):
    images = convert_from_path(pdf_path)
    text = ""

    for img in images:
        text += pytesseract.image_to_string(img)

    return text

# **Function to extract text from file**

In [ ]:
def extract_text(file_path):

    if not os.path.exists(file_path):
        raise FileNotFoundError(f'File not found: {file_path}')

    file_type = get_file_type(file_path)

    if file_type == "image":
        return extract_from_image(file_path)

    elif file_type == "pdf":
        text = extract_text_pdf(file_path)

        if text.strip() == "":
            text = extract_scanned_pdf(file_path)

        return text

    else:
        return "Unsupported file type"

In [ ]:
!apt-get install poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


# Examples

In [ ]:
extract_text('/content/sample.pdf')

'Sample PDF\nThis is a simple PDF file. Fun fun fun.\nLorem ipsum dolor sit amet, consectetuer adipiscing elit. Phasellus facilisis odio sed mi.\nCurabitur suscipit. Nullam vel nisi. Etiam semper ipsum ut lectus. Proin aliquam, erat eget\npharetra commodo, eros mi condimentum quam, sed commodo justo quam ut velit.\nInteger a erat. Cras laoreet ligula cursus enim. Aenean scelerisque velit et tellus.\nVestibulum dictum aliquet sem. Nulla facilisi. Vestibulum accumsan ante vitae elit. Nulla\nerat dolor, blandit in, rutrum quis, semper pulvinar, enim. Nullam varius congue risus.\nVivamus sollicitudin, metus ut interdum eleifend, nisi tellus pellentesque elit, tristique\naccumsan eros quam et risus. Suspendisse libero odio, mattis sit amet, aliquet eget,\nhendrerit vel, nulla. Sed vitae augue. Aliquam erat volutpat. Aliquam feugiat vulputate nisl.\nSuspendisse quis nulla pretium ante pretium mollis. Proin velit ligula, sagittis at, egestas a,\npulvinar quis, nisl.\nPellentesque sit amet lec

In [ ]:
extract_text('/content/im.jpg')

'€€ = ~=HEADLINE\n\nLorem ipsum dolor sit amet, consectetur\nadipiscing elit, sed do eiusmod tempor\nincididunt ut labore et dolore magna aliqua.\nUt enim ad minim veniam, quis nostrud\nexercitation ullamco laboris nisi ut aliquip\nex ea commodo consequat. Duis aute irure\ndolor in reprehenderit in voluptate velit\nesse cillum dolore eu fugiat nulla pariatur.\nExcepteur sint occaecat cupidatat non\nproident, sunt in culpa qui officia deserunt\n\nmollit anim id est laborum af\n\n \n\x0c'

In [ ]:
extract_text('/content/scansmpl.pdf')

'No.\n\nTHE SLEREXE COMPANY LIMITED\n\nSAPORS LANE - BOOLE - DORSET - BH 25 8ER\nTELEPHONE BOOLE (945 13) 51617 - TELEX 123456\n\nOur Ref. 350/PJC/EAC 18th January, 1972.\n\nDr. P.N. Cundall,\nMining Surveys Ltd.,\nHolroyd Road,\nReading,\n\nBerks.\n\nDear Pete,\n\nPermit me to introduce you to the facility of facsimile\ntransmission.\n\nIn facsimile a photocell is caused to perform a raster scan over\nthe subject copy. The variations of print density on the document\ncause the photocell to generate an analogous electrical video signal.\nThis signal is used to modulate a carrier, which is transmitted to a\nremote destination over a radio or cable communications link.\n\nAt the remote terminal, demodulation reconstructs the video\nsignal, which is used to modulate the density of print produced by a\nprinting device. This device is scanning in a raster scan synchronised\nwith that at the transmitting terminal. As a result, a facsimile\ncopy of the subject document is produced.\n\nProbabl

# **Text Preprocessing**

In [ ]:
!pip install scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_sm-0.5.1.tar.gz

  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_sm-0.5.1.tar.gz (15.9 MB)
  Preparing metadata (setup.py) ... done
  Using cached spacy-3.4.4.tar.gz (1.2 MB)
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

### Cleaning

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9.,:/% ]', '', text)
    return text

### Tokenization

In [ ]:
def tokenize_text(text):

    tokens = word_tokenize(text)

    return tokens

### Remove punctuation

In [ ]:
def remove_punctuation(text):

    translator = str.maketrans('', '', string.punctuation)

    cleaned_text = text.translate(translator)

    return cleaned_text

### Remove stopwords

In [ ]:
from nltk.corpus import stopwords
_base_stopwords = set(stopwords.words('english'))
# Preserve medical negation words — critical for medical meaning
MEDICAL_STOP_WORDS = _base_stopwords - {'no', 'not', 'without', 'never', 'nor', 'none', 'against'}
```

---

### 3. `simplify_text()` returns a tuple but it's used incorrectly
`simplify_text()` returns `(simplified_text, explanations)` but the pipeline code (once fixed) needs to unpack it properly. The current call `sentence = explain_medical_terms(sentence)` ignores this entirely.

---

### 4. `extract_medical_entities()` returns wrong format
Your function appends only `ent.text` (strings), but the **actual output shown in your cell** is:
```
[('Patient suffers', 'ENTITY'), ('diabetes', 'ENTITY'), ...]

def remove_stopwords(text):
    """Remove stop words while preserving medical negation words."""
    words = word_tokenize(text)
    filtered = [w for w in words if w.lower() not in MEDICAL_STOP_WORDS]
    return ' '.join(filtered)


# Generally not needed because words like no,not,without change the meaning of medical reports.
# And we need these words

**Combined Function for punct and stopwords**

In [ ]:
def clean_nlp_text(text):

    text = remove_punctuation(text)

    text = remove_stopwords(text)

    return text

**Lemmatization**

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):

    return [lemmatizer.lemmatize(word) for word in tokens]

# **Preprocessing Pipeline**

In [ ]:
def medical_preprocess(text):

    text = clean_text(text)

    text = remove_punctuation(text)

    text = remove_stopwords(text)

    tokens = tokenize_text(text)

    tokens = lemmatize_tokens(tokens)

    return tokens

# **preprocess for summary**

In [ ]:
def preprocess_for_summary(text):
    """Lightweight preprocessing for the summarizer (preserves sentence structure)."""
    text = re.sub(r'\s+', ' ', text)   # collapse whitespace
    text = re.sub(    r'\n+', ' ', text)   # remove newlines
    sentences = [s.strip() for s in sent_tokenize(text) if s.strip()]
    # Deduplicate sentences
    seen = set()
    unique = [s for s in sentences if not (s.lower() in seen or seen.add(s.lower()))]
    return ' '.join(unique) if unique else None

# **Medical Entity Recognition**

In [ ]:
pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached blis-0.7.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.0/865.0 kB 39.6 MB/s eta 0:00:00
Using cached blis-0.7.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (10.2 MB)
  Created wheel for en_core_sci_sm: filename=en_core_sci_sm-0.5.4-py3-none-any.whl size=14778488 sha256=cc6d09525496722e15b143901f829a38e3b5209c1bc993aedf6fc1074f4d220d
  Stored in directory: /root/.cache/pip/wheels/49/7f/0f/ec0fc3a935bfe55e6ef2ca04b7a31e33cbd533a6d7cbd9e11e
Successfully built en_core_sci_sm
  Attempting uninstall: blis
    Found existing installation: blis 1.3.3
    Uninstalling blis-1.3.3:
      Successfully uninstalled

In [ ]:
import spacy

nlp = spacy.load("en_core_sci_sm")

/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [ ]:
def extract_medical_entities(text):

    doc = nlp(text)

    entities = []

    for ent in doc.ents:
        entities.append(ent.text)

    return list(set(entities))

In [ ]:
text = "Patient suffers from diabetes and hypertension."

print(extract_medical_entities(text))

[('Patient suffers', 'ENTITY'), ('diabetes', 'ENTITY'), ('hypertension', 'ENTITY')]


# **Simplification Function**

In [ ]:
def simplify_text(text, entities):

    simplified_text = text

    explanations = []

    for entity in entities:

        key = entity.lower()

        if key in MEDICAL_EXPLANATIONS:

            explanation = MEDICAL_EXPLANATIONS[key]

            explanations.append(f"{entity} means {explanation}")

            simplified_text = simplified_text.replace(
                entity,
                f"{entity} ({explanation})"
            )

    return simplified_text, explanations

# **Sentence Simplification using LLM**

In [ ]:
pip install transformers

In [ ]:
from transformers import pipeline

simplifier = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

def simplify_sentence(sentence):

    prompt = f"Explain this medical sentence in simple words: {sentence}"

    result = simplifier(prompt, max_length=120)

    return result[0]['generated_text']

# **Full Simplifier Pipeline**

In [ ]:
def simplify_medical_report(text):
    sentences = sent_tokenize(text)
    simplified_sentences = []

    for sentence in sentences:
        entities = extract_medical_entities(sentence)
        sentence, _ = simplify_text(sentence, entities)
        simple_sentence = simplify_sentence(sentence)
        simplified_sentences.append(simple_sentence)

    return " ".join(simplified_sentences)